# 오늘 학습 정리 - OpenCV 기하학적 변환

---

## 1. 마크다운 표 정리

**Q. 동일한 분류인 Smoothing 같은 애들은 하나만 쓸 수 있나?**

마크다운은 셀 병합이 지원되지 않아서 첫 번째 행에만 분류를 쓰고 나머지는 비워두는 방식으로 표현 가능. HTML 표나 Word/Excel을 사용하면 셀 병합 가능.

---

## 2. 수식 마크다운

**Q. 회전 변환 행렬을 마크다운으로 표현하면?**

```math
\begin{bmatrix} x' \\ y' \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix}
```

---

## 3. 마크다운 이미지 삽입

**Q. 마크다운에 이미지를 넣을 수 있나?**

```markdown
![대체텍스트](이미지URL)
![로고](./images/logo.png)
```

- 경로에 **공백이 있으면** 이미지가 렌더링 안 됨 → 공백을 `%20`으로 치환하거나 폴더명 공백 제거
- 이미지 크기 조절은 HTML 사용: `<img src="이미지URL" width="300">`

---

## 4. Perspective 변환

**Q. Perspective 변환이란?**

카메라 시점 변화나 원근감으로 인해 왜곡된 이미지를 다른 시점으로 변환하는 기법. 직선은 유지되지만 평행선은 유지되지 않음.

| 항목 | Affine 변환 | Perspective 변환 |
|------|------------|-----------------|
| 필요한 점 | 3쌍 | 4쌍 |
| 평행선 유지 | ✅ | ❌ |
| 변환 행렬 | 2×3 | 3×3 |
| 자유도 | 6 | 8 |

**Q. w'로 나누는 이유는?**

Perspective 변환은 동차 좌표계(Homogeneous Coordinates)를 사용. 행렬곱 결과로 나온 x', y', w'는 동차 좌표 상태이므로, 실제 2D 픽셀 좌표로 변환하려면 w'로 나눠야 함.

$$x_{final} = \frac{x'}{w'}, \quad y_{final} = \frac{y'}{w'}$$

나누기는 행렬곱 자체에 포함된 게 아니라 **별도로 수행**하는 연산. `cv2.warpPerspective()`가 내부적으로 이 두 단계를 자동 처리.

**Q. Perspective 변환의 자유도가 왜 8인가?**

- 3×3 행렬의 원소 개수 = 9개
- w'로 나누는 과정에서 행렬 전체에 임의의 상수 k를 곱해도 결과가 동일 → h₂₂ = 1로 고정 가능
- 따라서 자유롭게 결정할 수 있는 원소 = **8개**
- 점 1쌍 = 방정식 2개 → 4쌍 × 2 = 8개 방정식으로 8개 미지수 풀기 가능

---

## 5. cv2.warpAffine

**Q. cv2.warpAffine이란?**

```python
dst = cv2.warpAffine(src, M, dsize, flags, borderMode, borderValue)
```

| 매개변수 | 설명 |
|---------|------|
| `src` | 입력 이미지 |
| `M` | 2×3 변환 행렬 |
| `dsize` | 출력 이미지 크기 `(w, h)` |
| `flags` | 보간법 |
| `borderMode` | 경계 처리 방식 |
| `borderValue` | 경계 채울 색상 |

**Q. Affine 변환 행렬에서 1의 의미는?**

```
M = [[1, 0, 100],
     [0, 1, 100]]
```

- a₀₀ = 1: x 방향 스케일 1배 (크기 변화 없음)
- a₁₁ = 1: y 방향 스케일 1배 (크기 변화 없음)
- 100: x, y 방향으로 100픽셀 평행이동

---

## 6. Shear(기울기) 변환

**Q. Shear 변환에서 기울어지는 원리는?**

```
M = [[1, 0.3, 0],
     [0, 1,   0]]
```

계산: x' = x + 0.3y, y' = y

이미지 좌표계에서 y는 위에서 아래로 증가하기 때문에:
- y=0 (맨 위): x 이동 없음
- y=100 (중간): x가 30 밀림
- y=200 (아래): x가 60 밀림

→ 위는 고정, 아래로 갈수록 오른쪽으로 밀려서 기울어짐

---

## 7. 회전 변환

**Q. 복합 변환에서 회전 방향이 시계방향으로 되는 이유?**

수학 좌표계에서는 양수 각도 = 반시계 방향이지만, 이미지 좌표계는 y축이 아래 방향이라 반전됨.

그런데 `cv2.getRotationMatrix2D`는 이미지 좌표계의 y축 반전을 이미 고려해서 설계되어 있어서:
- `+45` → **반시계방향**
- `-45` → **시계방향**

---

## 8. Canny 엣지 검출

**Q. Canny 검출은 밝은 곳을 검출하는 건가?**

아니요, **변화량(Gradient)** 을 검출하는 것. 어두운 배경에 어두운 물체라도 경계에서 밝기가 변하면 검출 가능.

**Canny 동작 과정:**
1. 가우시안 블러 (노이즈 제거)
2. Sobel로 Gradient(변화량) 계산
3. Non-Maximum Suppression (엣지 얇게)
4. 이중 임계값(Hysteresis)으로 엣지 확정

**Q. 임계값 50, 150의 의미는?**

| 범위 | 처리 |
|------|------|
| 변화량 < 50 | ❌ 엣지 아님 |
| 50 ≤ 변화량 < 150 | 🔶 약한 엣지 (강한 엣지와 연결된 경우만 인정) |
| 변화량 ≥ 150 | ✅ 강한 엣지 |

→ 50, 150은 픽셀 밝기가 아니라 **Gradient(변화량) 크기**

---

## 9. Perspective 변환 좌표 설정

**Q. src_pts를 따로 만들어주는 이유는?**

`cv2.getPerspectiveTransform(src_pts, dst_pts)`가 "어디서 → 어디로" 쌍(pair)을 요구하기 때문. src_pts 없이는 "어디 있던 점인지" 알 수 없어 변환 행렬 계산 불가.

**Q. src_pts는 거의 항상 네 꼭짓점으로 쓰면 되나?**

이미지 전체를 변환할 때는 네 꼭짓점 사용:
```python
src_pts = np.array([[0, 0], [width, 0], [width, height], [0, height]])
```

일부 영역만 변환하거나 여백을 주려면 안쪽 좌표 사용:
```python
src_pts = np.array([[50,50], [width-50,50], [width-50,height-50], [50,height-50]])
```

---

## 10. Plastic Surgery Filter (왜곡 필터)

**Q. 왜곡 원리는?**

```
이동할 좌표 = 중심점 + (원래 거리 * factor)
```

- **중심점**: 이미지 정중앙 또는 지정한 좌표
- **delta**: 각 픽셀 - 중심점 (중심 기준 상대 거리)
- **distance**: 중심점에서 각 픽셀까지 직선 거리
- **factor**: `(distance / radius) ** strength`

**Q. factor를 제곱하는 이유는?**

0~1 사이 값에 1보다 큰 수(strength)를 제곱하면 비선형적 왜곡 가능:

| strength | 효과 |
|----------|------|
| > 1 | 중심 근처 왜곡 강하게 |
| = 1 | 균일한 왜곡 |
| < 1 | 가장자리 왜곡 강하게 |

**Q. delta를 쓰는 이유는?**

delta 없이 계산하면 이미지 좌상단(0,0) 기준으로 계산되어 왜곡 중심이 엉뚱한 곳이 됨. delta를 쓰면 좌표계를 중심점 기준으로 이동시켜 올바른 왜곡 적용 가능.

**Q. map_x[mask]를 수정하면 map_x 자체가 바뀌나?**

네, `map_x[mask] = 새값`은 map_x 자체를 수정하는 것. 마스크에 해당하는 부분만 바꾸고 나머지는 그대로 유지됨. 따라서 `cv2.remap()`에 map_x 전체를 넣으면 마스크 안팎이 합쳐진 완성된 좌표 지도가 전달됨.

---

## 11. enumerate()

**Q. enumerate()란?**

인덱스와 값을 동시에 꺼내주는 파이썬 내장 함수.

```python
fruits = ['apple', 'banana', 'cherry']

for i, fruit in enumerate(fruits):
    print(i, fruit)
# 0 apple
# 1 banana
# 2 cherry

# start 옵션으로 시작 인덱스 변경
for i, fruit in enumerate(fruits, start=1):
    print(i, fruit)
# 1 apple ...
```

---

